# QuantJourney SDK - Data Contract, Lineage and Audit Pattern

This notebook demonstrates a QuantJourney SDK workflow that shows how to retain provider, route, warnings, request metadata and source evidence next to normalized data outputs.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


In [ ]:
symbol = 'AAPL'
calls = {'prices': qj.eod.get_historical_prices(symbol=symbol, start_date='2024-01-01', end_date=END), 'ratios': qj.fmp.get_financial_ratios_ttm(symbol=symbol), 'filings': qj.sec.get_company_filings(symbol=symbol, limit=10), 'identity': qj.openfigi.get_figi_data(symbol=symbol, exchange='US')}


In [ ]:
def audit_row(name: str, payload: Any, provider: str, route: str) -> dict[str, Any]:
    value = unwrap(payload)
    rows = as_rows(payload)
    meta = value.get('meta', {}) if isinstance(value, dict) and isinstance(value.get('meta'), dict) else {}
    return {'dataset': name, 'provider': provider, 'route': route, 'available': payload is not None, 'rows': len(rows), 'shape': type(value).__name__, 'request_id': meta.get('request_id'), 'warnings': meta.get('warnings', [])}
audit = pd.DataFrame([audit_row('prices', calls['prices'], 'eod', 'equity.pricing.get_historical_prices'), audit_row('ratios', calls['ratios'], 'fmp', 'equity.fundamentals.get_financial_ratios_ttm'), audit_row('filings', calls['filings'], 'sec', 'regulatory.sec.get_company_filings'), audit_row('identity', calls['identity'], 'openfigi', 'reference.identifiers.get_figi_data')])
display(audit)


In [ ]:
prices = pd.DataFrame(as_rows(calls['prices']))
filings = pd.DataFrame(as_rows(calls['filings']))
if prices.empty:
    raise RuntimeError('No price data returned')
prices['date'] = pd.to_datetime(prices['date'], errors='coerce')
prices['price'] = pd.to_numeric(prices.get('adjusted_close', prices.get('close')), errors='coerce')
prices = prices.dropna(subset=['date', 'price']).set_index('date').sort_index()
ax = prices['price'].tail(252).plot(title='AAPL evidence timeline: price with filing events')
if not filings.empty:
    date_col = next((col for col in filings.columns if 'date' in str(col).lower()), None)
    if date_col:
        event_dates = pd.to_datetime(filings[date_col], errors='coerce').dropna()
        for event_date in event_dates.tail(8):
            ax.axvline(event_date, color='tab:red', alpha=0.35)
plt.ylabel('price')
plt.show()
evidence_packet = {'symbol': symbol, 'datasets': audit.to_dict(orient='records')}
print(json.dumps(evidence_packet, indent=2, default=str)[:2000])


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.